# RL Experiment 08: Hyperparameter Sensitivity

**Self-contained experiment notebook using DRY architecture.**

## Architecture (DRY Principle)

| Location | What | Example |
|----------|------|---------|
| `schedule_engine/notebooks/` | Reusable functions | `train_agent()`, `evaluate_agent()` |
| `src/schedule_engine/rl/` | Production RL components | PPO agent with configurable hyperparams |
| **This notebook** | Experiment-specific config | Learning rate sweep |

## Experiment Overview
- **Agent**: PPO with varying learning rates
- **Goal**: Analyze sensitivity to learning rate hyperparameter
- **Sweep**: 1e-4, 3e-4, 1e-3
- **Metrics**: Best fitness and convergence per learning rate

## 1. Imports (from `schedule_engine/notebooks/`)

In [ ]:
from __future__ import annotations
from datetime import datetime
from pathlib import Path

# DRY IMPORTS FROM schedule_engine/notebooks/
from schedule_engine.notebooks import (
    build_notebook_config,
    create_env,
    evaluate_agent,
    load_context,
    set_global_seed,
    train_agent,
)

print(" All imports from schedule_engine/notebooks/ successful!")

## 2. Configuration (Inline - Experiment-Specific)

In [ ]:

# RL EXPERIMENT 08 CONFIGURATION - Hyperparameter Sensitivity


SEED = 42
POP_SIZE = 20
MAX_GENERATIONS = 40
MAX_STEPS = 15
TIMESTEPS = 3000

# Learning rate sweep
LEARNING_RATES = [1e-4, 3e-4, 1e-3]

# Paths - Organized by experiment with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/notebooks/rl_08_hyperparam_{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Config: pop={POP_SIZE}, ngen={MAX_GENERATIONS}, timesteps={TIMESTEPS}")
print(f" Learning rates to test: {LEARNING_RATES}")
print(f" Output: {OUTPUT_DIR}")

## 3. Load Data

In [ ]:
# Set reproducibility
set_global_seed(SEED)

# Build config and load scheduling context
config = build_notebook_config(seed=SEED, overrides={"pop_size": POP_SIZE})
_, context = load_context(DATA_DIR, config)

print(f" Scheduling context loaded")

## 4. Learning Rate Sweep

In [ ]:
# Run sweep over learning rates
sweep_results = []

for lr in LEARNING_RATES:
    print(f"\nTesting learning_rate={lr:.0e}...")
    
    # Create fresh environment for each run
    env = create_env(
        context=context,
        pop_size=POP_SIZE,
        max_generations=MAX_GENERATIONS,
        max_steps=MAX_STEPS,
    )
    
    # Train with this learning rate
    agent, train_time = train_agent(
        agent_type="ppo",
        env=env,
        timesteps=TIMESTEPS,
        seed=SEED,
        learning_rate=lr,
    )
    
    # Evaluate
    result = evaluate_agent(agent, env, max_generations=MAX_GENERATIONS)
    
    sweep_results.append({
        "learning_rate": lr,
        "train_time": train_time,
        "best_fitness": result.best_fitness,
        "convergence_gen": result.convergence_gen,
    })
    
    print(f"  lr={lr:.0e}: best={result.best_fitness}, conv={result.convergence_gen}")

## 5. Results Summary

In [ ]:
print(f"\n{'='*60}")
print(f"RL EXPERIMENT 08: HYPERPARAMETER SENSITIVITY RESULTS")
print(f"{'='*60}")
print(f"\nLearning Rate Sweep Results:")
print(f"{'lr':>10s} | {'fitness':>10s} | {'conv_gen':>10s} | {'train_time':>10s}")
print(f"{'-'*45}")
for r in sweep_results:
    print(f"{r['learning_rate']:.0e} | {r['best_fitness']:>10.2f} | {r['convergence_gen']:>10d} | {r['train_time']:>10.2f}s")

# Find best learning rate
best_result = min(sweep_results, key=lambda x: x["best_fitness"])
print(f"\n Best learning rate: {best_result['learning_rate']:.0e} (fitness={best_result['best_fitness']})")
print(f"{'='*60}")

## 6. Save Results

In [ ]:
import json

# Save experiment results
results_data = {
    "experiment": "rl_08_hyperparameter_sensitivity",
    "timestamp": TIMESTAMP,
    "config": {
        "seed": SEED,
        "pop_size": POP_SIZE,
        "max_generations": MAX_GENERATIONS,
        "max_steps": MAX_STEPS,
        "timesteps": TIMESTEPS,
        "learning_rates": LEARNING_RATES,
    },
    "results": {
        "sweep_results": sweep_results,
        "best_learning_rate": best_result["learning_rate"],
        "best_fitness": best_result["best_fitness"],
    },
}

results_path = OUTPUT_DIR / "results.json"
with open(results_path, "w") as f:
    json.dump(results_data, f, indent=2)

print(f" Results saved to: {results_path}")